# EEG-GNN — run the real model + website on Colab

Run the cells top to bottom (Shift+Enter). You need two files from your PC / lab server:

1. **`webapp.zip`** — `D:\EEG-GNN-Seizure-Detection-main\eeg-gnn\webapp.zip`
2. **`best_model_v4.pt`** — from the lab server: `/home/pmilab/EEG_GNN_Project/results/omni_gat_v4/best_model_v4.pt`

Optional: **Runtime → Change runtime type → T4 GPU** makes analysis much faster.

⚠️ Research demo only. Only upload anonymised or public recordings: files go to Google's servers.

In [ ]:
# 1) Upload webapp.zip and best_model_v4.pt (select both in the file picker)
from google.colab import files
up = files.upload()
print('Uploaded:', list(up))

In [ ]:
# 2) Unpack and put the model in place
import os, shutil, zipfile, glob
assert os.path.exists('webapp.zip'), 'webapp.zip missing - rerun cell 1'
pt = [f for f in os.listdir('.') if f.startswith('best_model_v4') and f.endswith('.pt')]
assert pt, 'best_model_v4.pt missing - rerun cell 1'
shutil.rmtree('webapp', ignore_errors=True)
zipfile.ZipFile('webapp.zip').extractall('.')
os.makedirs('webapp/models', exist_ok=True)
shutil.copy(pt[0], 'webapp/models/best_model_v4.pt')
print(os.listdir('webapp'), os.listdir('webapp/models'))

In [ ]:
# 3) Install what Colab doesn't already have (torch + scikit-learn are preinstalled)
!pip -q install torch_geometric mne fastapi "uvicorn[standard]" python-multipart

In [ ]:
# 4) Self-test: load YOUR model and run it on the synthetic test file (no website yet)
import sys, time
sys.path.insert(0, 'webapp/api')
import pipeline
model, scaler, meta = pipeline.load_model('webapp/models/best_model_v4.pt', 'webapp/models/scaler.pkl')
print('Model loaded on', pipeline.DEVICE, '| checkpoint info:', meta)
t = time.time()
res = pipeline.run('webapp/test_data/synthetic_test.edf', model, scaler, max_windows=40)
print(f"Analysed {res['info']['windows_analysed']} windows, {res['info']['channels_used']} channels in {time.time()-t:.1f}s")
print('\nTop 8 electrodes (synthetic file: RT1-RT3 have the seizure-like bursts):')
for c in res['channels'][:8]:
    print(f"  {c['name']:6s} {c['prob']*100:5.1f}%")

In [ ]:
# 5) Start the API + website in the background
import subprocess, time, urllib.request, json
server = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'],
                          cwd='webapp/api', stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(json.load(urllib.request.urlopen('http://127.0.0.1:7860/health'))); break
    except Exception:
        time.sleep(1)
else:
    print(open('server.log').read())

In [ ]:
# 6) Get a public link (Cloudflare quick tunnel, no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re, subprocess, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n🔗 Open your site:', url, '\n(It may take ~20 s before the link responds. Keep this notebook open; the link dies when Colab stops.)')

In [ ]:
# (If something breaks) show the server log
print(open('server.log').read()[-4000:])